In [ ]:
import numpy as np
import pandas as pd

### Load datasets for modeling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

premodel_core = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Core_Final.csv")
premodel_gust = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Gust_Final.csv")

Mounted at /content/drive


/tmp/ipython-input-486/274579847.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  premodel_gust = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Gust_Final.csv")


### Quick dataset checks

In [ ]:
print("Core Shape:", premodel_core.shape)
print("Gust Shape:", premodel_gust.shape)
print("Core Class Balance Check:", premodel_core["High_Intensity"].value_counts(normalize=True))
print("Gust Class Balance Check:", premodel_gust["High_Intensity"].value_counts(normalize=True))

Core Shape: (1563080, 21)
Gust Shape: (1011445, 25)
Core Class Balance Check: High_Intensity
0    0.900896
1    0.099104
Name: proportion, dtype: float64
Gust Class Balance Check: High_Intensity
0    0.886772
1    0.113228
Name: proportion, dtype: float64


In [ ]:
premodel_core["LC_Type1"] = premodel_core["LC_Type1"].astype("category")
premodel_gust["LC_Type1"] = premodel_gust["LC_Type1"].astype("category")

print(premodel_core.dtypes)
print(premodel_gust.dtypes)


High_Intensity                     int64
Mean_Temp_C                      float64
Total_Precip_mm                  float64
LC_Type1                        category
Month                              int64
Day_of_Year                        int64
Latitude_Fire                    float64
Longitude_Fire                   float64
Elevation_m                      float64
FRP_max                          float64
Fire_ID                            int64
Climate_ID                        object
LC_Label                          object
Nearest_Core_Station_Dist_km     float64
Core_Dist_75km                      bool
Core_Dist_100km                     bool
Confidence_Ordered                 int64
DayNight                          object
Hour                               int64
Year                               int64
Province                          object
dtype: object
High_Intensity                     int64
Mean_Temp_C                      float64
Total_Precip_mm                  float64
Ma

# **1. RANDOM FOREST FOR CORE DATASET**

## **(1) Define Features and Response**

In [ ]:
# Core feature set for baseline model
core_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year",
    "LC_Type1"
]

X = premodel_core[core_features].copy()
y = premodel_core["High_Intensity"].copy()

### **(2) Define Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

# Startified split to ensure rare high-intensity class representation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

### **(3) Feature Adjustment Preprocessing**

In [ ]:
# Define feature types (numerical and categorical)
numeric_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year"
]

categorical_features = ["LC_Type1"]

### Construct preprocessing and model pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# No scaling for numeric, only encoding for land cover
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Input random forest model parameters
RF = RandomForestClassifier(
    n_estimators=300,
    random_state=123,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=5
)

# Define pipeline
pipeline = Pipeline(
    steps=[("preprocess", preprocess), ("model", RF)]
)

### **(4) Fit Model & Evaluate Performance**

In [ ]:
# Fit the model
pipeline.fit(X_train, y_train)

# Evaluate Performance
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC–AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.90      0.93    281634
           1       0.44      0.70      0.54     30982

    accuracy                           0.88    312616
   macro avg       0.70      0.80      0.74    312616
weighted avg       0.91      0.88      0.89    312616

ROC–AUC: 0.9035124416010074
Confusion Matrix:
 [[253839  27795]
 [  9165  21817]]


### **(5) Feature Importance Table**

In [ ]:
# Get feature names after preprocessing
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_

FI = (pd.DataFrame({"Feature": feature_names, "Importance": importances})
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True))

FI.head(20)

,Feature,Importance
0,num__Longitude_Fire,0.295915
1,num__Latitude_Fire,0.295732
2,num__Mean_Temp_C,0.138248
3,num__Day_of_Year,0.137618
4,num__Elevation_m,0.052382
5,num__Month,0.027670
6,num__Total_Precip_mm,0.025702
7,cat__LC_Type1_8,0.006393
8,cat__LC_Type1_1,0.005274
9,cat__LC_Type1_5,0.003805


# **2. RANDOM FOREST FOR GUST DATASET**

## **(1) Define Features and Response**

In [ ]:
# Gust feature set for baseline model
gust_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Max_Gust_Speed_kmh",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year",
    "LC_Type1"
]

X = premodel_gust[gust_features].copy()
y = premodel_gust["High_Intensity"].copy()

### **(2) Define Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

# Startified split to ensure rare high-intensity class representation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

### **(3) Feature Adjustment Preprocessing**

In [ ]:
# Define feature types (numerical and categorical)
numeric_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Max_Gust_Speed_kmh",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year"
]

categorical_features = ["LC_Type1"]

### Construct preprocessing and model pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# No scaling for numeric, only encoding for land cover
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Input random forest model parameters
RF = RandomForestClassifier(
    n_estimators=300,
    random_state=123,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=5
)

# Define pipeline
pipeline = Pipeline(
    steps=[("preprocess", preprocess), ("model", RF)]
)

### **(4) Fit Model & Evaluate Performance**

In [ ]:
# Fit the model
pipeline.fit(X_train, y_train)

# Evaluate Performance
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC–AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.89      0.92    179384
           1       0.44      0.71      0.55     22905

    accuracy                           0.87    202289
   macro avg       0.70      0.80      0.73    202289
weighted avg       0.90      0.87      0.88    202289

ROC–AUC: 0.8936132016533177
Confusion Matrix:
 [[159141  20243]
 [  6687  16218]]


### **(5) Feature Importance Table**

In [ ]:
# Get feature names after preprocessing
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_

FI = (pd.DataFrame({"Feature": feature_names, "Importance": importances})
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True))

FI.head(20)

,Feature,Importance
0,num__Longitude_Fire,0.295671
1,num__Latitude_Fire,0.291752
2,num__Mean_Temp_C,0.108154
3,num__Day_of_Year,0.101845
4,num__Max_Gust_Speed_kmh,0.068531
5,num__Elevation_m,0.051478
6,num__Total_Precip_mm,0.029275
7,num__Month,0.023840
8,cat__LC_Type1_8,0.007699
9,cat__LC_Type1_1,0.006376
